# Aula 10 · Interpolação polinomial

Esta aula apresenta o [capítulo 10 do site](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/). A ideia central: **por n pontos passa um único polinômio de grau n − 1**, e ele estima os valores entre os pontos de uma tabela — mas nunca fora dela.

**Ao fim da aula você consegue:**

1. interpolar linearmente entre dois pontos de uma tabela;
2. construir o polinômio interpolador por Vandermonde, Lagrange e Newton;
3. calcular diferenças divididas e a base de Lagrange à mão;
4. reconhecer que extrapolar é perigoso e que a escolha da variável importa.

**Roteiro:** 🧩 · 1. entre os pontos · 2. Vandermonde · 3. 🧑‍🏫 Lagrange · 4. 🧑‍🏫 Newton · 5. extrapolar · 6. outra área · 🎯 prática · 🧩 o termistor · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Eletrônica embarcada — o termômetro que mentia.**
>
> *O termômetro digital de uma estufa usa um **termistor**: um resistor cuja
> resistência cai quando a temperatura sobe. O microcontrolador mede a resistência e
> converte em temperatura por uma tabela de 5 pontos do datasheet. O estagiário
> programou um polinômio que passa pelos 5 pontos, e agora, numa tarde amena, o
> termômetro marca **54 °C**. "**O polinômio passa exatamente pela tabela — como
> pode estar tão errado?**"*

No fim da aula, você descobre o erro e conserta a conversão — sem mudar o método,
só a variável.

## 1. Estimar entre os pontos

O catálogo de um cabo coaxial dá a atenuação em 5 frequências. E em 300 MHz?

📖 [capítulo 10 · Estimar entre os pontos](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#estimar-entre-os-pontos)

In [ ]:
# 📦 dados prontos — só rode esta célula
# Catálogo de um cabo coaxial: atenuação (dB/100 m) em algumas frequências (MHz)
freq = [50.0, 100.0, 200.0, 400.0, 1000.0]
aten = [4.48, 6.40, 9.17, 13.20, 21.61]

**✍️ Passo 1.** Faça a interpolação linear em 300 MHz entre os pontos de 200 e 400 MHz (posições 2 e 3): a inclinação entre eles, e o valor em 300.

In [ ]:
# ✍️ passo 1

**Preveja:** vai dar mais perto de 9,17 ou de 13,20?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

11,185 dB/100 m: quase no meio, porque 300 está no meio de 200 e 400. A reta
ignora que a curva é **curva** — e o valor verdadeiro é um pouco diferente.

📖 [capítulo 10 · Estimar entre os pontos](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#estimar-entre-os-pontos)

</details>

### 🎯 Sua vez — A reta entre dois pontos

Escreva `reta_entre(x0, y0, x1, y1, x)`, que devolve o valor em `x` da reta que liga $(x_0, y_0)$ a $(x_1, y_1)$.

In [ ]:
def reta_entre(x0, y0, x1, y1, x):
    # sua solução aqui
    pass

In [ ]:
confere(reta_entre, [
    ((200, 9.17, 400, 13.20, 300), 11.184999999999999),
    ((0, 5, 10, 25, 2), 9.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Inclinação $(y_1 - y_0)/(x_1 - x_0)$, e o valor $y_0 + \text{inclinação}\cdot(x - x_0)$.

</details>

## 2. Um polinômio por todos os pontos

Por 5 pontos passa um único polinômio de grau 4. Cada ponto dá uma equação
$a_0 + a_1x_i + \cdots + a_4x_i^4 = y_i$: um sistema linear, com a **matriz de
Vandermonde** ($V_{ij} = x_i^j$).

📖 [capítulo 10 · Um polinômio por todos os pontos](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#um-polinomio-por-todos-os-pontos)

**✍️ Passo 2.** Monte `V = np.zeros((5, 5))` com `V[i, j] = freq[i] ** j` (dois laços), resolva `coef = np.linalg.solve(V, np.array(aten))` e calcule o polinômio em 300 (a soma de `coef[j] * 300**j`).

In [ ]:
# ✍️ passo 2

**Preveja:** vai dar o mesmo que a reta?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

11,232 dB/100 m — um pouco acima da reta, porque o polinômio acompanha a
curvatura. Os coeficientes vão de $2$ a $10^{-10}$: com mais pontos, a matriz
de Vandermonde fica mal condicionada (capítulo 7).

📖 [capítulo 10 · Um polinômio por todos os pontos](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#um-polinomio-por-todos-os-pontos)

</details>

## 3. No quadro: o polinômio de Lagrange

📖 [capítulo 10 · No quadro: o polinômio de Lagrange](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#no-quadro-o-polinomio-de-lagrange)

### 🧑‍🏫 No quadro — a base de Lagrange

Caderno de papel aberto. No quadro:

1. a peça $L_i(x)$ que vale 1 em $x_i$ e 0 nos outros pontos;
2. o produto que faz isso: $\prod_{j\neq i} (x - x_j)/(x_i - x_j)$;
3. somar $y_i L_i(x)$;
4. à mão, com os pontos $(1, 2)$, $(2, 3)$ e $(4, 1)$, em $x = 3$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ p(x) = \sum_i y_i\,L_i(x), \qquad L_i(x) = \prod_{j \neq i} \frac{x - x_j}{x_i - x_j} $$

Com $(1,2), (2,3), (4,1)$: $L_0(3) = -\tfrac13$, $L_1(3) = 1$, $L_2(3) = \tfrac13$,
e $p(3) = 2{,}6667$.

</details>

### 🎯 Sua vez — Uma peça da base

Escreva `base_lagrange(xs, i, x)`, que devolve $L_i(x)$.

In [ ]:
def base_lagrange(xs, i, x):
    # sua solução aqui
    pass

In [ ]:
confere(base_lagrange, [
    (([1, 2, 4], 0, 3), -0.3333333333333333),
    (([1, 2, 4], 2, 3), 0.3333333333333333),
    (([1, 2, 4], 1, 2), 1.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Comece com `L = 1.0` e multiplique pelo fator de cada `j` diferente de `i`.

</details>

**✍️ Passo 3.** Com a sua `base_lagrange`, calcule $p(300) = \sum_i$ `aten[i] * base_lagrange(freq, i, 300)` num laço.

In [ ]:
# ✍️ passo 3

**Preveja:** vai dar o mesmo que o Vandermonde?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Exatamente o mesmo: 11,232. Por 5 pontos passa **um único** polinômio de grau 4 —
Vandermonde e Lagrange são dois caminhos até ele.

📖 [capítulo 10 · No quadro: o polinômio de Lagrange](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#no-quadro-o-polinomio-de-lagrange)

</details>

## 4. No quadro: as diferenças divididas de Newton

Altura média de meninas (curva da OMS, aproximada): 0, 6, 12, 24 e 36 meses; 49,1;
65,7; 74,0; 86,4 e 95,1 cm. Quanto mede uma menina de 18 meses?

📖 [capítulo 10 · No quadro: as diferenças divididas de Newton](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#no-quadro-as-diferencas-divididas-de-newton)

### 🧑‍🏫 No quadro — o polinômio de Newton

Caderno de papel aberto. No quadro:

1. a forma $c_0 + c_1(x - x_0) + c_2(x - x_0)(x - x_1) + \cdots$;
2. $c_0 = y_0$, $c_1$ = a inclinação dos dois primeiros pontos;
3. a tabela de diferenças divididas, coluna por coluna;
4. um ponto novo acrescenta só um termo.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$f[x_i, x_{i+1}] = \dfrac{y_{i+1} - y_i}{x_{i+1} - x_i}$,
$f[x_i, x_{i+1}, x_{i+2}] = \dfrac{f[x_{i+1}, x_{i+2}] - f[x_i, x_{i+1}]}{x_{i+2} - x_i}$, ...

Os coeficientes $c_k$ são a primeira linha da tabela.

</details>

**✍️ Passo 4.** Com `idade = [0, 6, 12, 24, 36]` e `altura = [49.1, 65.7, 74.0, 86.4, 95.1]`, faça `coef = altura.copy()` e, `for ordem in range(1, 5):`, `for i in range(4, ordem - 1, -1):`, `coef[i] = (coef[i] - coef[i - 1]) / (idade[i] - idade[i - ordem])`. Imprima `coef`.

In [ ]:
# ✍️ passo 4

**Preveja:** quanto vale `coef[1]`, e o que ele significa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`2.7667`: a inclinação entre os dois primeiros pontos — a menina cresce 2,77 cm por mês no primeiro semestre.

</details>

**✍️ Passo 5.** Calcule o polinômio de Newton em 18 meses: `p = coef[0]`, `produto = 1.0` e, para `k` de 1 a 4, `produto = produto * (18 - idade[k - 1])` e `p = p + coef[k] * produto`.

In [ ]:
# ✍️ passo 5

**Preveja:** a estimativa fica perto dos 80,7 cm que a OMS publica para 18 meses?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

80,0 cm: a menos de 1 cm, com 5 pontos na tabela.

📖 [capítulo 10 · No quadro: as diferenças divididas de Newton](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#no-quadro-as-diferencas-divididas-de-newton)

</details>

## 5. Interpolar não é extrapolar

📖 [capítulo 10 · Interpolar não é extrapolar](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#interpolar-nao-e-extrapolar)

In [ ]:
# 📦 dados prontos — só rode esta célula
def lagrange(xs, ys, x):
    soma = 0.0
    for i in range(len(xs)):
        L = 1.0
        for j in range(len(xs)):
            if j != i:
                L = L * (x - xs[j]) / (xs[i] - xs[j])
        soma = soma + ys[i] * L
    return soma

**✍️ Passo 6.** Calcule `lagrange(freq, aten, f)` para `f` = 300, 1500 e 2000 MHz. Compare com o modelo físico do cabo, `0.62 * np.sqrt(f) + 0.002 * f`.

In [ ]:
# ✍️ passo 6

**Preveja:** o polinômio acerta fora da tabela (que vai até 1000 MHz)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Em 300, erra 0,1 dB. Em 1500 e 2000 MHz, dá **−80** e **−568** dB: atenuação
negativa, um cabo que amplificaria o sinal. Fora da tabela, o polinômio faz o que
os termos de grau alto mandam. **Nunca extrapole.**

📖 [capítulo 10 · Interpolar não é extrapolar](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#interpolar-nao-e-extrapolar)

</details>

## 6. Mesmo método, outra área

**Animação.** O artista define a altura de um personagem só em alguns quadros-chave
(0, 10, 20 e 30, com 0; 1,8; 2,4 e 0 m) e o programa preenche o resto.

📖 [capítulo 10 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Com a `lagrange`, imprima a altura nos quadros 0, 5, 10, ..., 30.

In [ ]:
# ✍️ passo 7

**Preveja:** no quadro 15, a altura fica entre 1,8 e 2,4?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

2,36 m: dentro, e a curva do pulo sai suave. A mesma função que estimou a
atenuação de um cabo preenche os quadros de uma animação.

📖 [capítulo 10 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *5. Interpolar não é extrapolar*.
📖 [capítulo 10 · Interpolar não é extrapolar](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/10-interpolacao-polinomial/#interpolar-nao-e-extrapolar)

### 🎯 Sua vez — Quanto o polinômio erra?

Escreva `erro_maximo(xs, ys, f, pontos)`, que devolve o **maior** erro absoluto
entre `lagrange(xs, ys, x)` e a função verdadeira `f(x)`, entre todos os `x` da
lista `pontos`.

In [ ]:
def erro_maximo(xs, ys, f, pontos):
    # sua solução aqui
    pass

In [ ]:
def quadrado(x):
    return x**2


confere(erro_maximo, [
    (([0.0, 1.0, 2.0], [0.0, 1.0, 4.0], quadrado, [0.5, 1.5, 3.0]), 0.0),
    (([0.0, 1.0], [0.0, 1.0], quadrado, [0.5]), 0.25),
])

<details>
<summary><b>💡 Dica</b></summary>

Padrão extremo sobre `pontos`, com `abs(lagrange(...) - f(x))`. (Uma parábola por 3 pontos de $x^2$ é o próprio $x^2$: o erro é zero.)

</details>

## 🧩 Resolvendo o problema

> *"**O polinômio passa exatamente pela tabela — como pode estar tão errado?**"* — o estagiário.

A célula 📦 tem a tabela do datasheet e a fórmula física do termistor (a "equação
beta"), só para conferir.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Tabela do datasheet de um termistor NTC de 10 kΩ: temperatura (°C) e resistência (Ω)
T_tab = [0.0, 25.0, 50.0, 75.0, 100.0]
R_tab = [32650.0, 10000.0, 3603.0, 1481.0, 678.0]


def formula_fabricante(R):
    """O modelo físico do termistor (equação "beta", B = 3950), para conferir."""
    return 1 / (1 / 298.15 + np.log(R / 10000) / 3950) - 273.15

Primeiro, o que o estagiário fez: Lagrange com a **resistência** como $x$, numa leitura de 7500 Ω.

In [ ]:
print("Lagrange em R:   ", lagrange(R_tab, T_tab, 7500.0), "°C")
print("fórmula do fabricante:", formula_fabricante(7500.0), "°C")

<details>
<summary><b>▶ Por que tanto erro?</b></summary>

Os valores de resistência vão de 678 a 32 650 Ω, e quase todos os pontos estão
espremidos no começo (abaixo de 4000). Entre 10 000 e 32 650 Ω, o polinômio de grau
4 não tem ponto nenhum para "segurar" a curva, e oscila. É o mesmo fenômeno da
extrapolação, dentro da tabela.

O termistor segue $1/T \propto \ln R$: no **logaritmo** da resistência, os pontos
ficam bem espaçados e a curva é quase uma reta. Interpolar em $\ln R$, e não em
$R$, muda tudo.

</details>

### 🎯 Sua vez — A conversão consertada

Escreva `temperatura_ntc(R)`, que interpola a temperatura com a `lagrange`, mas
usando `np.log` da resistência como $x$: monte a lista dos logaritmos de
`R_tab` e chame a `lagrange` com `np.log(R)`.

In [ ]:
def temperatura_ntc(R):
    # sua solução aqui
    pass

In [ ]:
confere(temperatura_ntc, [
    ((7500.0,), 31.690574013017958),
    ((2000.0,), 66.17416144229026),
])

<details>
<summary><b>💡 Dica</b></summary>

Um laço com `append(np.log(r))` para cada `r` de `R_tab`; depois `lagrange(ln_tab, T_tab, np.log(R))`.

</details>

In [ ]:
for R in [7500.0, 5000.0, 2000.0]:
    print(R, "Ω:", "ln R ->", temperatura_ntc(R), "| R ->", lagrange(R_tab, T_tab, R),
          "| fabricante ->", formula_fabricante(R))

<details>
<summary><b>▶ O que os números dizem</b></summary>

Em 7500 Ω, o polinômio em $R$ dizia 54,3 °C; em $\ln R$, **31.69 °C**, a 0,07 °C
da fórmula do fabricante (31.62 °C). O **método é o mesmo**
— o mesmo Lagrange, os mesmos 5 pontos. O que mudou foi a **variável**: numa escala em
que a curva é quase reta, qualquer interpolação acerta.

Essa ideia — transformar os dados até a relação ficar simples — volta com força no
capítulo 15 (ajuste não linear), com o nome de **linearização**.

</details>

## 📋 A lista

Abra a [Lista 10](https://lacouth.github.io/metodos_telecom-site/listas/lista10/). O **Exercício 01** é à mão (✏️): Lagrange e diferenças
divididas por 3 pontos. Comece por ele, no papel.

**a)** Quanto vale $L_1(x)$ nos pontos $x_0 = 1$ e $x_2 = 4$? E em $x_1 = 2$?

<details>
<summary><b>▶ Resposta</b></summary>

$L_1(1) = 0$, $L_1(4) = 0$ e $L_1(2) = 1$: é para isso que a peça foi construída.

</details>

Termine o exercício e siga para o **Exercício 02**, a interpolação linear em qualquer tabela.

## 🚪 Antes de sair

**1.** Vandermonde, Lagrange e Newton dão polinômios diferentes para a mesma tabela?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Não: por $n$ pontos passa um único polinômio de grau $n - 1$. Os três chegam a ele por caminhos diferentes.

</details>

**2.** Qual a vantagem de Newton quando chega um ponto novo na tabela?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Os coeficientes antigos continuam valendo: o ponto novo acrescenta só um termo ao polinômio.

</details>

**3.** Por que o polinômio do termistor funcionou com $\ln R$ e não com $R$?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque em $\ln R$ os pontos ficam bem espaçados e a relação com a temperatura é quase uma reta; em $R$, os pontos se amontoam num canto e o polinômio oscila no resto.

</details>

## 🏠 Para casa

- Refaça no papel a base de Lagrange e a tabela de diferenças divididas **sem olhar**.
- Termine a [Lista 10](https://lacouth.github.io/metodos_telecom-site/listas/lista10/).
- Leia o começo do [capítulo 11](https://lacouth.github.io/metodos_telecom-site/unidade5-interpolacao/11-runge-splines/): e se a
  tabela tiver 20 pontos?